In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import os
import time

In [9]:
# Constants
RANDOMSTATE = 1082
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "Logistic Regression"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [10]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [11]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOMSTATE, stratify=y
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
# Train the model
model = LogisticRegression(
    random_state=RANDOMSTATE,
    max_iter=2000,
    solver='saga'
)
start_time = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [13]:
# Predict and evaluate
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy on Test Set: 99.79%
F1 Score: 0.9979
Confusion Matrix:
[[2500    0]
 [   6  392]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [14]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(model.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../UAVATTACK_Results.csv
